In [ ]:
!pip install crewai crewai-tools langchain-ollama pywintypes -q

ERROR: Could not find a version that satisfies the requirement pywintypes (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pywintypes


In [3]:
import os
from crewai import Agent, Crew, Process, Task
from crewai_tools import FileWriterTool
from langchain_ollama import ChatOllama

# 1. Configure the Local LLM via Ollama
# We point to localhost where Ollama manages our open-source models
local_llm = ChatOllama(
    model="llama3", 
    base_url="http://localhost:11434"
)

# 2. Initialize Shared Automation Tools
# The File Writer allows the developer and DevOps agents to output clean code locally
file_writer_tool = FileWriterTool()

print("🤖 Initializing SDLC Multi-Agent Team...")

# 3. Define the Specialized Agents
pm_agent = Agent(
    role="Lead Product Manager",
    goal="Define the project MVP, core features, and tech stack specification.",
    backstory="You excel at taking abstract project ideas and refining them into concrete, scoped Markdown spec sheets while ruthlessly preventing scope creep.",
    llm=local_llm,
    verbose=True,
    memory=True
)

architect_agent = Agent(
    role="Lead Software Architect",
    goal="Design the internal system architecture, API endpoints, and database schema.",
    backstory="You translate business requirements into technical blueprints, ensuring scalable database models and clean code structure flows.",
    llm=local_llm,
    verbose=True
)

developer_agent = Agent(
    role="Senior Full-Stack Developer",
    goal="Write high-quality, modular, and runnable code based on system architecture designs.",
    backstory="A pragmatic software engineer who writes clean, secure, and idiomatic code with solid error handling.",
    tools=[file_writer_tool],
    llm=local_llm,
    verbose=True
)

qa_agent = Agent(
    role="QA Automation Engineer",
    goal="Write automated unit and integration tests to ensure code quality.",
    backstory="A security-minded engineer with a knack for finding edge cases, logical flaws, and writing test suites to break code.",
    llm=local_llm,
    verbose=True
)

devops_agent = Agent(
    role="DevOps Engineer",
    goal="Generate Dockerfiles, CI/CD configuration files, and setup deployment blueprints.",
    backstory="An expert in cloud deployments, infrastructure-as-code, and zero-downtime automated pipelines.",
    tools=[file_writer_tool],
    llm=local_llm,
    verbose=True
)


# 4. Define the Sequential Tasks
task_planning = Task(
    description="Analyze the user's project requirements: {project_description}. Generate a comprehensive Markdown document listing the MVP features, user flows, and tech stack constraints.",
    expected_output="A structured Markdown product specification sheet.",
    agent=pm_agent
)

task_design = Task(
    description="Review the PM specification document. Create a complete database data schema, architectural breakdown, and an explicit list of backend/frontend API routes.",
    expected_output="An engineering blueprint document detailing schemas, components, and data flows.",
    agent=architect_agent
)

task_implementation = Task(
    description="Using the architectural blueprints, write the foundational web app code files. Use the File Writer tool to save the primary code blocks into structured local directories.",
    expected_output="A working code repository structure containing the core application logic files.",
    agent=developer_agent
)

task_testing = Task(
    description="Analyze the source code generated by the developer. Generate a corresponding automated test suite handling standard execution, form validations, and logical edge cases.",
    expected_output="Functional automated test files aligned perfectly with the code implementation.",
    agent=qa_agent
)

task_deployment = Task(
    description="Review the final code architecture. Write a production-grade Dockerfile, a `.dockerignore` file, and a CI/CD workflow configuration (like GitHub Actions) to automate cloud deployment.",
    expected_output="Ready-to-use Docker and infrastructure pipeline configuration files.",
    agent=devops_agent
)


# 5. Assemble the Crew into a Sequential SDLC Workflow
sdlc_crew = Crew(
    agents=[pm_agent, architect_agent, developer_agent, qa_agent, devops_agent],
    tasks=[task_planning, task_design, task_implementation, task_testing, task_deployment],
    process=Process.sequential  # Hand-off pipeline: Planning -> Design -> Implement -> Test -> Deploy
)

# 6. Kickoff the System for your specific project
if __name__ == "__main__":
    # Define your project intent here
    inputs = {
        "project_description": "A Linux Commands Helper tool where users type natural language requests, and it generates the correct shell command along with safe flags and usage explanations."
    }
    
    print("\n🚀 Kickstarting the Agent Pipeline execution...")
    final_output = sdlc_crew.kickoff(inputs=inputs)
    
    print("\n=================== SYSTEM WORKFLOW COMPLETE ===================")
    print("Your multi-agent team has finalized the project lifecycle cycle.")

ModuleNotFoundError: No module named 'pywintypes'